In [1]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")


Using device: mps


In [2]:
model_name = "textattack/distilbert-base-uncased-MRPC"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

print(f"Loaded model: {model_name}")
print(f"Number of labels: {model.config.num_labels}")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loaded model: textattack/distilbert-base-uncased-MRPC
Number of labels: 2


In [3]:
dataset = load_dataset("glue", "mrpc", split="validation")
labels_all = np.array(dataset["label"])

target_total = 200
classes, counts = np.unique(labels_all, return_counts=True)
proportions = counts / counts.sum()
base_counts = np.floor(proportions * target_total).astype(int)
remainder = target_total - base_counts.sum()
fractional = proportions * target_total - base_counts
order = np.argsort(-fractional)
for i in range(remainder):
    base_counts[order[i]] += 1

selected_indices = []
class_selection_summary = {}
for cls, needed in zip(classes, base_counts):
    cls_indices = np.where(labels_all == cls)[0]
    chosen = cls_indices[:needed]
    selected_indices.extend(chosen.tolist())
    class_selection_summary[int(cls)] = {
        "available": int(len(cls_indices)),
        "selected": int(len(chosen)),
    }

selected_indices = sorted(selected_indices)
subset = dataset.select(selected_indices)
labels = np.array(subset["label"])

print("Dataset split: glue/mrpc validation")
print(f"Original examples: {len(dataset)}")
print(f"Deterministic stratified subset size: {len(subset)}")
print("Class selection summary:")
print(class_selection_summary)
print("First subset row:")
print(subset[0])


Dataset split: glue/mrpc validation
Original examples: 408
Deterministic stratified subset size: 200
Class selection summary:
{0: {'available': 129, 'selected': 63}, 1: {'available': 279, 'selected': 137}}
First subset row:
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}


In [4]:
pair_lengths = []
for s1, s2 in zip(subset["sentence1"], subset["sentence2"]):
    encoded = tokenizer(s1, s2, truncation=False, add_special_tokens=True)
    pair_lengths.append(len(encoded["input_ids"]))

pair_lengths = np.array(pair_lengths)
quantiles = np.quantile(pair_lengths, [0.0, 1/3, 2/3, 1.0])
bucket_edges = [int(np.floor(quantiles[0])), int(np.floor(quantiles[1])), int(np.floor(quantiles[2])), int(np.ceil(quantiles[3]))]

if bucket_edges[1] <= bucket_edges[0]:
    bucket_edges[1] = bucket_edges[0] + 1
if bucket_edges[2] <= bucket_edges[1]:
    bucket_edges[2] = bucket_edges[1] + 1
if bucket_edges[3] <= bucket_edges[2]:
    bucket_edges[3] = bucket_edges[2] + 1

def length_bucket(length):
    if length < bucket_edges[1]:
        return "short"
    elif length < bucket_edges[2]:
        return "medium"
    return "long"

bucket_names = np.array([length_bucket(x) for x in pair_lengths])

print("Length bucket edges:")
print({
    "short": f"[{bucket_edges[0]}, {bucket_edges[1]})",
    "medium": f"[{bucket_edges[1]}, {bucket_edges[2]})",
    "long": f"[{bucket_edges[2]}, {bucket_edges[3]}]",
})
print("Bucket counts:")
unique_buckets, bucket_counts = np.unique(bucket_names, return_counts=True)
print({k: int(v) for k, v in zip(unique_buckets, bucket_counts)})


Length bucket edges:
{'short': '[29, 47)', 'medium': '[47, 59)', 'long': '[59, 84]'}
Bucket counts:
{np.str_('long'): 72, np.str_('medium'): 62, np.str_('short'): 66}


In [5]:
batch_size = 32
predictions = []
confidences = []
positive_probs = []
full_probs = []

for start_idx in range(0, len(subset), batch_size):
    batch = subset[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)
        confs = probs.max(dim=-1).values

    predictions.extend(preds.cpu().tolist())
    confidences.extend(confs.cpu().tolist())
    positive_probs.extend(probs[:, 1].cpu().tolist())
    full_probs.extend(probs.cpu().tolist())

predictions = np.array(predictions)
confidences = np.array(confidences)
positive_probs = np.array(positive_probs)
full_probs = np.array(full_probs)

print(f"Completed inference for {len(predictions)} examples.")


Completed inference for 200 examples.


In [6]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", zero_division=0)

overall_metrics = {
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
}

bucket_metrics = {}
for bucket in ["short", "medium", "long"]:
    mask = bucket_names == bucket
    count = int(mask.sum())
    if count == 0:
        bucket_metrics[bucket] = {
            "count": 0,
            "accuracy": None,
            "precision": None,
            "recall": None,
            "f1": None,
            "avg_confidence": None,
            "avg_pair_length": None,
        }
        continue

    y_true = labels[mask]
    y_pred = predictions[mask]
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    bucket_metrics[bucket] = {
        "count": count,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(p),
        "recall": float(r),
        "f1": float(f),
        "avg_confidence": float(confidences[mask].mean()),
        "avg_pair_length": float(pair_lengths[mask].mean()),
    }

print("Overall metrics:")
for k, v in overall_metrics.items():
    print(f"{k:>10}: {v:.4f}")

print("\nPer-length-bucket metrics:")
for bucket in ["short", "medium", "long"]:
    info = bucket_metrics[bucket]
    print(f"bucket={bucket}")
    for key, value in info.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.4f}")
        else:
            print(f"  {key}: {value}")


Overall metrics:
  accuracy: 0.8650
 precision: 0.8481
    recall: 0.9781
        f1: 0.9085

Per-length-bucket metrics:
bucket=short
  count: 66
  accuracy: 0.8939
  precision: 0.8696
  recall: 0.9756
  f1: 0.9195
  avg_confidence: 0.8737
  avg_pair_length: 39.4394
bucket=medium
  count: 62
  accuracy: 0.7742
  precision: 0.7547
  recall: 0.9756
  f1: 0.8511
  avg_confidence: 0.8805
  avg_pair_length: 52.7419
bucket=long
  count: 72
  accuracy: 0.9167
  precision: 0.9153
  recall: 0.9818
  f1: 0.9474
  avg_confidence: 0.9082
  avg_pair_length: 68.0278


In [7]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}

short_indices = np.where(bucket_names == "short")[0]
long_indices = np.where(bucket_names == "long")[0]

shortest_idx = short_indices[np.argmin(pair_lengths[short_indices])] if len(short_indices) > 0 else None
longest_idx = long_indices[np.argmax(pair_lengths[long_indices])] if len(long_indices) > 0 else None

def print_bucket_example(local_idx, title):
    row = subset[int(local_idx)]
    true_label = int(labels[local_idx])
    pred_label = int(predictions[local_idx])
    print(title)
    print(f"subset_index    : {local_idx}")
    print(f"original_index  : {selected_indices[local_idx]}")
    print(f"bucket          : {bucket_names[local_idx]}")
    print(f"pair_length     : {int(pair_lengths[local_idx])}")
    print(f"sentence1       : {row['sentence1']}")
    print(f"sentence2       : {row['sentence2']}")
    print(f"true_label      : {true_label} ({label_map[true_label]})")
    print(f"pred_label      : {pred_label} ({label_map[pred_label]})")
    print(f"confidence      : {float(confidences[local_idx]):.4f}")
    print(f"p(class=0)      : {float(full_probs[local_idx][0]):.4f}")
    print(f"p(class=1)      : {float(positive_probs[local_idx]):.4f}")
    print("-" * 80)

if shortest_idx is not None:
    print_bucket_example(shortest_idx, "Example from shortest bucket")
else:
    print("No example available in shortest bucket.")

if longest_idx is not None:
    print_bucket_example(longest_idx, "Example from longest bucket")
else:
    print("No example available in longest bucket.")


Example from shortest bucket
subset_index    : 76
original_index  : 76
bucket          : short
pair_length     : 29
sentence1       : McCabe said he was considered a witness , not a suspect .
sentence2       : " He is not considered a suspect , " McCabe said .
true_label      : 0 (not_paraphrase)
pred_label      : 1 (paraphrase)
confidence      : 0.9726
p(class=0)      : 0.0274
p(class=1)      : 0.9726
--------------------------------------------------------------------------------
Example from longest bucket
subset_index    : 120
original_index  : 120
bucket          : long
pair_length     : 84
sentence1       : Three such vigilante-style attacks forced the hacker organizer , who identified himself only as " Eleonora [ 67 ] , " to extend the contest until 7 p.m. EST Sunday .
sentence2       : Three such vigilante-style attacks forced the hacker organiser , who identified himself only as " Eleonora67 ] , " to extend the contest until 8am ( AEST ) today .
true_label      : 1 (paraphrase

In [8]:
result_summary = {
    "model": model_name,
    "dataset_split": "glue/mrpc validation",
    "subset_strategy": "deterministic stratified 200-example subset by label",
    "device": device,
    "num_examples": int(len(subset)),
    "length_bucket_edges": {
        "short": [int(bucket_edges[0]), int(bucket_edges[1])],
        "medium": [int(bucket_edges[1]), int(bucket_edges[2])],
        "long": [int(bucket_edges[2]), int(bucket_edges[3])],
    },
    "overall_metrics": overall_metrics,
    "bucket_metrics": bucket_metrics,
}

print("RESULT SUMMARY")
print(result_summary)


RESULT SUMMARY
{'model': 'textattack/distilbert-base-uncased-MRPC', 'dataset_split': 'glue/mrpc validation', 'subset_strategy': 'deterministic stratified 200-example subset by label', 'device': 'mps', 'num_examples': 200, 'length_bucket_edges': {'short': [29, 47], 'medium': [47, 59], 'long': [59, 84]}, 'overall_metrics': {'accuracy': 0.865, 'precision': 0.8481012658227848, 'recall': 0.9781021897810219, 'f1': 0.9084745762711864}, 'bucket_metrics': {'short': {'count': 66, 'accuracy': 0.8939393939393939, 'precision': 0.8695652173913043, 'recall': 0.975609756097561, 'f1': 0.9195402298850575, 'avg_confidence': 0.873675039320281, 'avg_pair_length': 39.43939393939394}, 'medium': {'count': 62, 'accuracy': 0.7741935483870968, 'precision': 0.7547169811320755, 'recall': 0.975609756097561, 'f1': 0.851063829787234, 'avg_confidence': 0.8805015836992571, 'avg_pair_length': 52.74193548387097}, 'long': {'count': 72, 'accuracy': 0.9166666666666666, 'precision': 0.9152542372881356, 'recall': 0.9818181818